In [ ]:
import os
import json
import joblib
import pandas as pd
import lightgbm as lgb

In [ ]:
path_train = "data\\res_data\\train.parquet"
path_test = "data\\res_data\\test.parquet"
path_train_target = "data\\train\\train_target.parquet"

path_models_old = "data\\model\\model_extra_feature2"
path_models_new = "data\\model\\after_optuna"

In [ ]:
train = pd.read_parquet(path_train)
train_t = pd.read_parquet(path_train_target)

test = pd.read_parquet(path_test)

In [ ]:
target_cols = [col for col in train_t.columns if col != "customer_id"]

In [ ]:
def load_target_artifact(target, artifacts_dir="optuna_artifacts"):
    target_dir = os.path.join(artifacts_dir, target)

    meta_path = os.path.join(target_dir, "meta.json")
    drop_path = os.path.join(target_dir, "feature_drop.json")

    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)

    with open(drop_path, "r", encoding="utf-8") as f:
        feature_drop = json.load(f)

    return meta, feature_drop


def build_train_test_for_target(train, test, train_t, target, artifacts_dir="optuna_artifacts"):
    meta, feature_drop = load_target_artifact(target, artifacts_dir)

    train_cur = train.drop(columns=feature_drop, errors="ignore").copy()
    test_cur = test.drop(columns=feature_drop, errors="ignore").copy()

    # на всякий случай выравниваем test под train
    missing_in_test = [col for col in train_cur.columns if col not in test_cur.columns]
    for col in missing_in_test:
        if str(train_cur[col].dtype) == "category":
            test_cur[col] = pd.Series(
                pd.Categorical([pd.NA] * len(test_cur), categories=train_cur[col].cat.categories),
                index=test_cur.index,
            )
        else:
            test_cur[col] = pd.NA

    extra_in_test = [col for col in test_cur.columns if col not in train_cur.columns]
    if extra_in_test:
        test_cur.drop(columns=extra_in_test, inplace=True, errors="ignore")

    if missing_in_test: print(f"Найдена разница по столбцам: {len(missing_in_test + extra_in_test)}")

    test_cur = test_cur[train_cur.columns]

    y = train_t[target].copy()

    return train_cur, y, test_cur, meta


def train_one_target_and_predict(
    train,
    test,
    train_t,
    target,
    artifacts_dir="optuna_artifacts",
    models_dir="final_models",
    save_model=True,
):
    os.makedirs(models_dir, exist_ok=True)

    train_cur, y, test_cur, meta = build_train_test_for_target(
        train=train,
        test=test,
        train_t=train_t,
        target=target,
        artifacts_dir=artifacts_dir,
    )

    params = meta["best_params"].copy()

    # пересчитываем scale_pos_weight уже на полном train
    pos = y.sum()
    neg = len(y) - pos
    scale_pos_weight = neg / max(pos, 1)
    scale_pos_weight = min(max(scale_pos_weight, 1.0), 100.0)
    params["scale_pos_weight"] = float(scale_pos_weight)

    model = lgb.LGBMClassifier(**params)
    model.fit(train_cur, y)

    preds = model.predict_proba(test_cur)[:, 1]

    if save_model:
        joblib.dump(model, os.path.join(models_dir, f"{target}.pkl"))

    return preds


def make_submission(
    train,
    test,
    train_t,
    target_cols,
    artifacts_dir="optuna_artifacts",
    models_dir="final_models",
    id_col="customer_id",
    submission_path="submission.csv",
):
    all_preds = {}

    for target in target_cols:
        print(f"Training {target} ...")
        preds = train_one_target_and_predict(
            train=train,
            test=test,
            train_t=train_t,
            target=target,
            artifacts_dir=artifacts_dir,
            models_dir=models_dir,
            save_model=True,
        )
        nums = target.split("_")[1:]
        all_preds[f"predict_{nums[0]}_{nums[1]}"] = preds
        print(f"{target} done")

    submission = pd.DataFrame(all_preds)

    if id_col in test.columns:
        submission.insert(0, id_col, test[id_col].values)

    submission.to_parquet(submission_path, index=False)

    return submission

In [ ]:
submission = make_submission(
    train=train,
    test=test,
    train_t=train_t,
    target_cols=target_cols,
    artifacts_dir="optuna_artifacts",
    models_dir="final_models",
    id_col="customer_id",
    submission_path="submission.csv",
)

In [ ]:
submission

predicts_10_1
